<a href="https://colab.research.google.com/github/Sathwik828/renifrocment-Learinig-Lab/blob/main/RL8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# ASSIGNMENT: STATE VALUE AND ACTION VALUE
# Using discounted returns without the Bellman equation
# ============================================================

# Grid World MDP (grid_env_det.py equivalent)

class GridWorld:
    def __init__(self):
        self.states = ["S0", "S1", "S2", "S3", "S4", "S5"]
        self.actions = ["up", "down", "left", "right"]

        self.rows = 2
        self.cols = 3
        self.start_state = "S0"
        self.goal_state = "S2"
        self.obstacle = "S4"
        self.gamma = 0.9

        self.rewards = {
            "normal": -1,
            "goal": 10,
            "obstacle": -5
        }

    def transition(self, state, action):
        # Goal is terminal
        if state == self.goal_state:
            return state

        index = int(state[1])
        row, col = index // self.cols, index % self.cols

        new_row, new_col = row, col

        if action == "up":
            new_row -= 1
        elif action == "down":
            new_row += 1
        elif action == "left":
            new_col -= 1
        elif action == "right":
            new_col += 1
        else:
            raise ValueError("Invalid action")

        # Boundary check
        if not (0 <= new_row < self.rows and
                0 <= new_col < self.cols):
            return state

        next_state = f"S{new_row * self.cols + new_col}"

        # Obstacle cannot be entered
        if next_state == self.obstacle:
            return state

        return next_state

    def step(self, state, action):
        next_state = self.transition(state, action)

        if next_state == self.goal_state:
            reward = self.rewards["goal"]
        elif next_state == state:
            index = int(state[1])
            row, col = index // self.cols, index % self.cols

            nr, nc = row, col
            if action == "up":
                nr -= 1
            elif action == "down":
                nr += 1
            elif action == "left":
                nc -= 1
            elif action == "right":
                nc += 1

            if (0 <= nr < self.rows and 0 <= nc < self.cols
                    and f"S{nr * self.cols + nc}" == self.obstacle):
                reward = self.rewards["obstacle"]
            else:
                reward = self.rewards["normal"]
        else:
            reward = self.rewards["normal"]

        return next_state, reward


# Create environment
env = GridWorld()

# Given policy
policy = {
    "S0": "right",
    "S1": "down",
    "S3": "right",
    "S4": "up"
}

# Complete policy for states not specified in the question
policy["S2"] = "right"  # Goal state; terminal
policy["S5"] = "up"     # Additional state

# Calculate state value by following the policy
# No Bellman equation is used.
def state_value_function(state, max_steps=100):
    total_return = 0
    discount = 1.0
    current_state = state

    for _ in range(max_steps):
        if current_state == env.goal_state:
            break

        action = policy[current_state]
        next_state, reward = env.step(current_state, action)

        total_return += discount * reward
        discount *= env.gamma
        current_state = next_state

    return total_return


# Calculate action value by taking the specified action first,
# then following the policy for subsequent steps.
def action_value_function(state, action, max_steps=100):
    total_return = 0
    discount = 1.0
    current_state = state

    # First action is the action given as input
    next_state, reward = env.step(current_state, action)
    total_return += reward
    discount *= env.gamma
    current_state = next_state

    # Follow policy after the first action
    for _ in range(max_steps - 1):
        if current_state == env.goal_state:
            break

        next_action = policy[current_state]
        next_state, reward = env.step(current_state, next_action)

        total_return += discount * reward
        discount *= env.gamma
        current_state = next_state

    return total_return


# Display results
print("=" * 50)
print("STATE VALUE AND ACTION VALUE")
print("=" * 50)

print("\nGiven Policy:", policy)
print("Discount Factor (gamma):", env.gamma)

v_s0 = state_value_function("S0")
q_s0_up = action_value_function("S0", "up")

print("\nState Value V(S0):", round(v_s0, 4))
print("Action Value Q(S0, up):", round(q_s0_up, 4))


STATE VALUE AND ACTION VALUE

Given Policy: {'S0': 'right', 'S1': 'down', 'S3': 'right', 'S4': 'up', 'S2': 'right', 'S5': 'up'}
Discount Factor (gamma): 0.9

State Value V(S0): -45.9987
Action Value Q(S0, up): -42.3987
